In [ ]:
import jax
import jax.numpy as jnp
import torch
import numpy as np

rng = np.random.default_rng(seed=666)

In [ ]:
x = rng.random((1024, 1024))
y = rng.random((1024, 1024))

NameError: name 'rng' is not defined

In [ ]:
x_jax = jnp.array(x, dtype=jnp.float32)
y_jax = jnp.array(y, dtype=jnp.float32)
o_jax_1 = jax.lax.dot(x_jax, y_jax, precision=jax.lax.Precision.DEFAULT)
o_jax_2 = jax.lax.dot(x_jax, y_jax, precision=jax.lax.Precision.HIGH)
o_jax_3 = jax.lax.dot(x_jax, y_jax, precision=jax.lax.Precision.HIGHEST)

In [ ]:
def compare_metrics(test_val, ref_val, name="Test"):
    # 确保转换为 float64 进行高精度对比
    test_val = test_val.astype(jnp.float64)
    ref_val = ref_val.astype(jnp.float64)

    abs_diff = jnp.abs(test_val - ref_val)

    max_abs = jnp.max(abs_diff)

    print(f"--- {name} vs Reference ---")
    print(f"Max Absolute Error: {max_abs:.6e}")
    print(f"RMSE: {jnp.sqrt(jnp.mean(jnp.square(abs_diff))):.6e}")

不等待耗时: 0.009136s
等待耗时: 1.034402s


In [ ]:
# 使用示例
compare_metrics(o_jax_1, o_jax_2, name="DEFAULT (o1 vs o2)")
compare_metrics(o_jax_2, o_jax_3, name="HIGH (o2 vs o3)")
compare_metrics(o_jax_1, o_jax_3, name="HIGHEST (o1 vs o3)")

In [ ]:
x_torch = torch.tensor(x, dtype=torch.float32).cuda()
y_torch = torch.tensor(y, dtype=torch.float32).cuda()
o_torch = torch.matmul(x_torch, y_torch).detach().cpu().numpy()
o_torch_1 = (
    torch.ops.aten.einsum.default("ij,jk->ik", [x_torch, y_torch])
    .detach()
    .cpu()
    .numpy()
)

In [ ]:
compare_metrics(o_torch, o_torch_1, name="PyTorch (o_torch) vs PyTorch (o_torch_1)")
compare_metrics(o_torch, o_jax_1, name="PyTorch (o_torch) vs JAX (o_jax_1_default)")
compare_metrics(o_torch, o_jax_2, name="PyTorch (o_torch) vs JAX (o_jax_2_high)")
compare_metrics(o_torch, o_jax_3, name="PyTorch (o_torch) vs JAX (o_jax_3_highest)")